# CSV Basics — 09: CSV to Parquet Migration

## What you will learn
Converting a CSV data lake to Parquet is one of the highest-ROI
data engineering tasks. This notebook shows you the complete,
production-ready migration process.

In this notebook:
1. Why migrate: size and speed comparison
2. Schema validation before migration
3. Choosing the right partition column
4. Incremental migration (new CSV files → Parquet)
5. Validation: verify row counts and checksums
6. Production migration pipeline with rollback


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 19:44:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


In [2]:
import pathlib, random, datetime, time, subprocess

random.seed(42)

CATEGORIES = ["Electronics","Clothing","Books","Food","Sports"]
COUNTRIES  = ["US","UK","DE","FR","JP"]

def make_csv_batch(n, start_id, year, month):
    """Generate one month of order data as CSV."""
    rows = ["order_id,customer_id,product,category,country,quantity,price,status,order_date"]
    for i in range(n):
        d = datetime.date(year, month, random.randint(1, 28))
        qty   = random.randint(1, 10)
        price = round(random.uniform(5, 500), 2)
        rows.append(
            f"{start_id+i},{random.randint(1,5000)},Product_{random.randint(1,100)},"
            f"{random.choice(CATEGORIES)},{random.choice(COUNTRIES)},"
            f"{qty},{price},{random.choice(['pending','confirmed','shipped','delivered'])},"
            f"{d}"
        )
    return "\n".join(rows)

# Create 6 monthly CSV files (simulate a data lake)
csv_dir = f"{DATA_DIR}/csv_lake"
pathlib.Path(csv_dir).mkdir(exist_ok=True)

total_rows = 0
for i, (year, month) in enumerate([(2024,1),(2024,2),(2024,3),(2024,4),(2024,5),(2024,6)]):
    n = random.randint(8000, 12000)
    content = make_csv_batch(n, total_rows+1, year, month)
    path = f"{csv_dir}/orders_{year}_{month:02d}.csv"
    pathlib.Path(path).write_text(content)
    total_rows += n
    size_kb = pathlib.Path(path).stat().st_size // 1024
    print(f"  {path.split('/')[-1]}: {n:,} rows, {size_kb} KB")

print(f"\nTotal: {total_rows:,} rows in {csv_dir}")

  orders_2024_01.csv: 10,619 rows, 624 KB
  orders_2024_02.csv: 8,645 rows, 517 KB
  orders_2024_03.csv: 8,753 rows, 524 KB
  orders_2024_04.csv: 9,551 rows, 571 KB
  orders_2024_05.csv: 8,011 rows, 479 KB
  orders_2024_06.csv: 10,874 rows, 650 KB

Total: 56,453 rows in /workspace/data/csv_basics/csv_lake


## Step 1 — Size and Speed Comparison


In [3]:
schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

# Read all CSV
df_csv = spark.read.csv(f"{csv_dir}/", header=True, schema=schema)
csv_count = df_csv.count()

# Convert to Parquet (partitioned by category)
pq_dir = f"{DATA_DIR}/parquet_lake"
df_csv.write.mode("overwrite").partitionBy("category").parquet(pq_dir)

# Size comparison
import os
csv_size_kb  = sum(p.stat().st_size for p in pathlib.Path(csv_dir).glob("*.csv")) // 1024
pq_size_kb   = sum(p.stat().st_size for p in pathlib.Path(pq_dir).rglob("*.parquet")) // 1024

print(f"Storage comparison ({csv_count:,} rows):")
print(f"  CSV files    : {csv_size_kb:,} KB ({len(list(pathlib.Path(csv_dir).glob('*.csv')))} files)")
print(f"  Parquet files: {pq_size_kb:,} KB (partitioned by category)")
print(f"  Compression  : {csv_size_kb/pq_size_kb:.1f}x smaller")

Storage comparison (61,453 rows):
  CSV files    : 3,667 KB (7 files)
  Parquet files: 1,020 KB (partitioned by category)
  Compression  : 3.6x smaller


In [4]:
# Speed comparison
queries = [
    ("Full scan + sum",
     lambda: spark.read.csv(f"{csv_dir}/", header=True, schema=schema).agg(F.sum("price")).collect(),
     lambda: spark.read.parquet(pq_dir).agg(F.sum("price")).collect()),
    ("Category filter",
     lambda: spark.read.csv(f"{csv_dir}/", header=True, schema=schema).filter(F.col("category")=="Electronics").count(),
     lambda: spark.read.parquet(pq_dir).filter(F.col("category")=="Electronics").count()),
    ("GroupBy + agg",
     lambda: spark.read.csv(f"{csv_dir}/", header=True, schema=schema).groupBy("category").agg(F.sum("price")).collect(),
     lambda: spark.read.parquet(pq_dir).groupBy("category").agg(F.sum("price")).collect()),
]

print(f"{'Query':<30} {'CSV':>8} {'Parquet':>10} {'Speedup':>10}")
print("-" * 62)
for label, csv_fn, pq_fn in queries:
    tc, tp = [], []
    for _ in range(3):
        t0=time.time(); csv_fn(); tc.append(time.time()-t0)
        t0=time.time(); pq_fn();  tp.append(time.time()-t0)
    t_c, t_p = sorted(tc)[1], sorted(tp)[1]
    print(f"  {label:<28} {t_c:>6.3f}s  {t_p:>8.3f}s  {t_c/t_p:>8.1f}x")

Query                               CSV    Parquet    Speedup
--------------------------------------------------------------
  Full scan + sum               0.335s     1.000s       0.3x
  Category filter               0.287s     0.673s       0.4x
  GroupBy + agg                 0.287s     0.900s       0.3x


## Step 2 — Validation: Row Count and Checksum


In [5]:
def validate_migration(csv_path, parquet_path, schema, key_col="order_id"):
    """
    Validate that CSV → Parquet migration is complete and correct.
    Checks: row count, key uniqueness, numeric column sums.
    """
    df_csv = spark.read.csv(csv_path, header=True, schema=schema)
    df_pq  = spark.read.parquet(parquet_path)

    csv_count = df_csv.count()
    pq_count  = df_pq.count()

    print(f"Row count validation:")
    print(f"  CSV     : {csv_count:,}")
    print(f"  Parquet : {pq_count:,}")
    print(f"  Match   : {'✓ PASS' if csv_count == pq_count else '✗ FAIL'}")
    print()

    # Numeric column checksums
    numeric_cols = [f.name for f in schema.fields
                    if f.dataType in (DoubleType(), FloatType(), LongType(), IntegerType())]

    print("Numeric column checksums:")
    all_match = True
    for col in numeric_cols[:4]:  # check first 4
        try:
            csv_sum = df_csv.agg(F.sum(col)).collect()[0][0] or 0
            pq_sum  = df_pq.agg(F.sum(col)).collect()[0][0] or 0
            match   = abs(csv_sum - pq_sum) < 0.01
            all_match = all_match and match
            print(f"  {col:<20} CSV={csv_sum:>12.2f}  PQ={pq_sum:>12.2f}  {'✓' if match else '✗'}")
        except Exception:
            pass

    print()
    print(f"Overall: {'✓ MIGRATION VALID' if csv_count==pq_count and all_match else '✗ MIGRATION HAS ISSUES'}")
    return csv_count == pq_count and all_match

validate_migration(f"{csv_dir}/", pq_dir, schema)

Row count validation:
  CSV     : 61,453
  Parquet : 61,453
  Match   : ✓ PASS

Numeric column checksums:
  order_id             CSV=1888266331.00  PQ=1888266331.00  ✓
  customer_id          CSV=153467982.00  PQ=153467982.00  ✓
  quantity             CSV=   337966.00  PQ=   337966.00  ✓
  price                CSV= 15524142.58  PQ= 15524142.58  ✓

Overall: ✓ MIGRATION VALID


True

## Step 3 — Incremental Migration: New CSV → Parquet


In [6]:
# New CSV files arrive daily — convert only new ones
processed_log = f"{DATA_DIR}/.migration_log"

def get_processed_files():
    try:
        return set(pathlib.Path(processed_log).read_text().strip().split("\n"))
    except FileNotFoundError:
        return set()

def mark_processed(filename):
    with open(processed_log, "a") as f:
        f.write(filename + "\n")

def incremental_migrate(csv_dir, parquet_dir, schema):
    """Process only CSV files not yet migrated to Parquet."""
    all_files = sorted(pathlib.Path(csv_dir).glob("*.csv"))
    processed = get_processed_files()
    new_files  = [f for f in all_files if f.name not in processed]

    if not new_files:
        print("No new files to process.")
        return 0

    print(f"Processing {len(new_files)} new file(s):")
    total = 0
    for csv_file in new_files:
        df = spark.read.csv(str(csv_file), header=True, schema=schema)
        count = df.count()
        df.write.mode("append").partitionBy("category").parquet(parquet_dir)
        mark_processed(csv_file.name)
        print(f"  {csv_file.name}: {count:,} rows migrated")
        total += count
    return total

# Run 1: migrate all existing
print("=== Run 1: Initial migration ===")
n = incremental_migrate(csv_dir, f"{DATA_DIR}/incremental_pq", schema)
print(f"Total migrated: {n:,}")

# Simulate new file arriving
new_content = make_csv_batch(5000, total_rows+1, 2024, 7)
pathlib.Path(f"{csv_dir}/orders_2024_07.csv").write_text(new_content)

print("\n=== Run 2: New July file arrived ===")
n = incremental_migrate(csv_dir, f"{DATA_DIR}/incremental_pq", schema)
print(f"Total migrated: {n:,}")

print(f"\nFinal Parquet count: {spark.read.parquet(f'{DATA_DIR}/incremental_pq').count():,}")

=== Run 1: Initial migration ===
No new files to process.
Total migrated: 0

=== Run 2: New July file arrived ===
No new files to process.
Total migrated: 0

Final Parquet count: 61,453


## Summary: CSV → Parquet Migration Checklist

1. **Validate CSV first** — check for encoding issues, dirty data, schema consistency
2. **Define explicit schema** — don't rely on inferSchema during migration
3. **Choose partition column** — date or low-cardinality categorical (category, region)
4. **Validate after migration** — row counts + numeric checksums
5. **Run incremental** — process only new files using a processed log
6. **Keep CSV as archive** — don't delete source until Parquet is verified

```
Migration pipeline:
  CSV files → validate schema → clean → convert → validate → archive CSV
```
